# Introduction: What is Tokenization?

**Tokenization** is the fundamental first step in Natural Language Processing (NLP). It is the process of breaking down a continuous stream of text into smaller, discrete units called **tokens**.

### Why is it needed?

1.  **Computers speak numbers, not words:** Neural networks and machine learning models cannot understand raw text strings. They operate on mathematical vectors. Tokenization converts text into a sequence of atomic units that can be mapped to numerical IDs (indices in a vocabulary).
2.  **Handling Complexity:**
    *   **Word-level tokenization** (splitting by space) creates massive vocabularies and struggles with rare words (e.g., "unfriend" might be unknown even if "friend" is known).
    *   **Character-level tokenization** results in a small vocabulary but produces very long sequences that are harder for models to learn from.
    *   **Subword tokenization** (the focus of this notebook) finds a middle ground. It breaks rare words into meaningful sub-units (e.g., `tokenization` $\rightarrow$ `token` + `##ization`). This is the approach used by modern Large Language Models (LLMs) like BERT and GPT.

# Exercise: DFA for "Greedy" Tokenization (MaxMatch)

## Concept: Deterministic Finite Automata (DFA) & WordPiece

**The Task:** Given a small vocabulary of subwords (e.g., `{ "a", "ab", "abc", "d" }`), we will construct a mechanism that performs "MaxMatch" tokenization. This is similar to how WordPiece tokenizers work in models like BERT.

**The Challenge:** We must handle the "Lookahead" problem. If the input is `abcd`, the tokenizer should greedily match the longest possible token (`abc`) and then `d`, rather than stopping early at `a` or `ab`.

In [2]:
# Define our small vocabulary
vocab = ["a", "ab", "abc", "d"]
print(f"Vocabulary: {vocab}")

Vocabulary: ['a', 'ab', 'abc', 'd']


## Implementation: Trie as a State Machine

To implement the DFA efficiently, we can use a **Trie (Prefix Tree)**. Each node in the Trie represents a state in our automata.

1. **Transitions**: Moving from one character to the next follows the edges of the Trie.
2. **Accepting States**: Nodes that mark the end of a word in our vocabulary are "accepting states".
3. **Greedy Logic**: To solve the lookahead challenge, we traverse as deep as possible. We keep track of the *last valid accepting state* we passed through. If we hit a dead end, we commit to that longest match.

Check the following link for an interactive trie illustration:
https://www.cs.usfca.edu/~galles/visualization/Trie.html

## `TrieNode` Class

The `TrieNode` class represents a single node in the Trie (Prefix Tree) data structure. In our `MaxMatchTokenizer`, the Trie is used to efficiently store the vocabulary and perform lookups for subwords. Each `TrieNode` acts as a state in a Deterministic Finite Automaton (DFA), where transitions between states are made by following characters.

Here's a breakdown of its key member variables:

1.  **`self.children`**:
    *   **Type:** Dictionary (`dict`)
    *   **Purpose:** This dictionary maps characters to their corresponding child `TrieNode` objects. Each key in the dictionary is a character, and its value is another `TrieNode`. This structure allows us to traverse the Trie by following characters in the input text, forming paths that represent prefixes of words in the vocabulary.

2.  **`self.is_end_of_word`**:
    *   **Type:** Boolean (`bool`)
    *   **Purpose:** This flag indicates whether the path from the root of the Trie to the current `TrieNode` forms a complete, valid word (or subword) that exists in our vocabulary. If `True`, it means this node marks the end of a recognized token; otherwise, it's just an intermediate prefix.

3.  **`self.token`**:
    *   **Type:** String (`str`) or `None`
    *   **Purpose:** If `is_end_of_word` is `True`, this variable stores the actual word (or subword) that ends at this node. This is useful for retrieving the matched token once an end-of-word state is reached during tokenization.

In [5]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end_of_word = False
        self.token = None

## `MaxMatchTokenizer` Class

The `MaxMatchTokenizer` class is designed to perform tokenization using a greedy, longest-match strategy, similar to how WordPiece tokenizers function. It leverages a Trie data structure to efficiently find subword units from a predefined vocabulary.

Here's a breakdown of its key components:

1.  **`__init__(self, vocab)`**:
    *   **Purpose:** The constructor initializes the tokenizer. It creates the root of the Trie and then calls `build_trie` to populate it with the provided vocabulary.

2.  **`build_trie(self, vocab)`**:
    *   **Purpose:** This method constructs the Trie. For each `word` (or subword) in the `vocab` list, it traverses the Trie, adding new `TrieNode` objects as children if a character path doesn't exist. Once a full word is inserted, the `is_end_of_word` flag of the last node is set to `True`, and the `token` attribute is stored.

3.  **`tokenize(self, text)`**:
    *   **Purpose:** This is the core method for tokenizing an input `text` string.
    *   **Mechanism:** It iterates through the input text character by character. For each starting position `i`, it attempts to find the longest possible match from the vocabulary by traversing the Trie (DFA).
    *   **Greedy Matching:** It keeps track of the `last_matched_len`, which represents the length of the longest valid subword found so far from the current starting position. This ensures the "MaxMatch" principle.
    *   **Handling Unknowns (`<UNK>`)**: If no match is found for a character or a sequence of characters, it treats them as unknown and assigns an `<UNK:char>` token. This prevents the tokenizer from getting stuck and allows it to process any input, even out-of-vocabulary terms.

In [6]:
class MaxMatchTokenizer:
    def __init__(self, vocab):
        self.root = TrieNode()
        self.build_trie(vocab)

    # Builds a Trie from the given vocabulary
    def build_trie(self, vocab):
        for word in vocab:
            node = self.root
            for char in word:
                if char not in node.children:
                    node.children[char] = TrieNode()
                node = node.children[char]
            node.is_end_of_word = True
            node.token = word

    def tokenize(self, text):
        tokens = []
        i = 0
        n = len(text)

        while i < n:
            node = self.root
            last_matched_len = 0
            current_len = 0

            # Traverse the DFA (Trie) to find the longest prefix
            j = i
            while j < n and text[j] in node.children:
                node = node.children[text[j]]
                current_len += 1

                # If we hit an accepting state, record it as a candidate
                if node.is_end_of_word:
                    last_matched_len = current_len

                j += 1

            # If we found a match, add it and advance
            if last_matched_len > 0:
                token = text[i : i + last_matched_len]
                tokens.append(token)
                i += last_matched_len
            else:
                # Handle unknown characters (no transition available)
                tokens.append(f"<UNK:{text[i]}>")
                i += 1

        return tokens

## Example run

In [7]:
# Initialize the tokenizer with our vocabulary
tokenizer = MaxMatchTokenizer(vocab)

In [12]:
# Test with the challenge input
input_text = "abcdabd"
tokens = tokenizer.tokenize(input_text)

print(f"Input: '{input_text}'")
print(f"Tokens: {tokens}")

# Verify against expected behavior
# assert tokens == ['abc', 'd'], "Error: Should maximize match to 'abc' then 'd'"

Input: 'abcdabd'
Tokens: ['abc', 'd', 'ab', 'd']


# Optional Material

# Real-World Example: BERT's WordPiece

Now let's see how this works in practice using the **BERT** tokenizer. BERT uses a vocabulary of ~30,000 subwords and the same greedy "MaxMatch" principle.

**Key Difference:** Real WordPiece tokenizers distinguish between the start of a word and the continuation of a word. Continuation subwords are marked with `##` (e.g., `##ing`).

In [13]:
# Install the transformers library
!pip install transformers -q

In [14]:
from transformers import BertTokenizer

In [15]:

# Load the pre-trained BERT tokenizer
# This tokenizer has a fixed vocabulary trained on a massive corpus
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# A sentence with some complex/rare words
text = "Tokenization is surprisingly fascinating."

# Perform WordPiece tokenization
tokens = bert_tokenizer.tokenize(text)

print(f"Original Text: '{text}'")
print(f"Tokens: {tokens}")

print("\nAnalysis:")
print(f"1. 'tokenization' became -> {bert_tokenizer.tokenize('tokenization')}")
print("   (It greedily matched 'token', then found '##ization' in its vocab)")
print(f"2. 'subwords' (marked with ##) attach to the previous token.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Original Text: 'Tokenization is surprisingly fascinating.'
Tokens: ['token', '##ization', 'is', 'surprisingly', 'fascinating', '.']

Analysis:
1. 'tokenization' became -> ['token', '##ization']
   (It greedily matched 'token', then found '##ization' in its vocab)
2. 'subwords' (marked with ##) attach to the previous token.


### Why weren't words like "surprisingly" split?

WordPiece is efficient because it only splits words when necessary. "Surprisingly" is common enough to be in the vocabulary as a whole token.

Let's force the tokenizer to work harder by giving it:
1. A made-up word (`subwordification`)
2. A very long, complex word (`supercalifragilisticexpialidocious`)

In [16]:
# Test with words that are definitely NOT in the fixed vocabulary
rare_words = [
    "subwordification",  # A made-up term
    "supercalifragilisticexpialidocious" # A classic long word
]

print("Aggressive Subword Splitting Examples:\n")
for word in rare_words:
    tokens = bert_tokenizer.tokenize(word)
    print(f"'{word}'")
    print(f" -> {tokens}\n")

Aggressive Subword Splitting Examples:

'subwordification'
 -> ['sub', '##word', '##ification']

'supercalifragilisticexpialidocious'
 -> ['super', '##cal', '##if', '##rag', '##ilis', '##tic', '##ex', '##pia', '##lid', '##oc', '##ious']

